In [7]:
import os
import json
import pandas as pd

In [8]:
if "moved_up_dir" not in globals():
    # Code that should run once
    %cd ..
    moved_up_dir = True
else:
    print("Skipping — already executed this session.")

Skipping — already executed this session.


In [9]:
nodes_df = pd.read_csv('data/graphs/only_connected/nodes.csv')

In [10]:
edges_df = pd.read_csv('data/graphs/only_connected/edges.csv')

In [11]:
features_df = pd.read_csv('data/artist_base_features_5_years_only_collab.csv')

In [12]:
nodes_df.shape, edges_df.shape, features_df.shape

((86762, 40), (1426829, 3), (86762, 40))

In [17]:
nodes_df[nodes_df['artist_name'] == 'Taylor Swift']

,artist_mbid,artist_name,window_years,debut_date,window_cutoff_date,releases_total,releases_per_year,avg_days_between_releases,release_velocity_releases_per_day,release_velocity_releases_per_year,...,primary_genre,all_genres_str,artist_country,artist_region_city,years_active,debut_year,debut_decade,recency_index,primary_role,all_roles_str
27704,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,5,1928-01-01,1933-01-01,1,0.199918,NaN,NaN,NaN,...,ambient,"ambient, celtic, christmas, christmas carol, c...",United States,West Reading,97.919233,1928,1920,1.0,composer,"composer, editor, instrument, lyricist, perfor..."


In [19]:
edges_df.shape

(1426829, 3)

In [25]:
unique_mbids = set(edges_df["u"]).union(edges_df["v"])
known_mbids = set(features_df["artist_mbid"])
missing_mbids = unique_mbids - known_mbids

len(unique_mbids), len(known_mbids), len(missing_mbids)

(243233, 86762, 156471)

In [31]:
missing_to_template = {}

for row in edges_df.itertuples(index=False):
    u = row.u
    v = row.v

    # If u is missing but v is known, we can use v as template for u
    if u in missing_mbids and v in known_mbids and u not in missing_to_template:
        missing_to_template[u] = v

    # If v is missing but u is known, we can use u as template for v
    if v in missing_mbids and u in known_mbids and v not in missing_to_template:
        missing_to_template[v] = u

print(f"Can assign templates for {len(missing_to_template)} of {len(missing_mbids)} missing mbids.")

# 4) Build synthetic rows by copying the template rows
missing_with_template = [
    m for m in missing_mbids
    if m in missing_to_template
]

if missing_with_template:
    # 2) Get the corresponding template mbids
    template_mbids = [missing_to_template[m] for m in missing_with_template]

    # 3) Index features_df by artist_mbid once
    features_idx = features_df.set_index("artist_mbid")

    # Keep only templates that actually exist in features_df
    mask_existing = [t in features_idx.index for t in template_mbids]
    missing_with_template = [m for m, ok in zip(missing_with_template, mask_existing) if ok]
    template_mbids = [t for t, ok in zip(template_mbids, mask_existing) if ok]

    if template_mbids:
        # 4) Grab all template rows in one shot, preserving order
        base_rows = features_idx.loc[template_mbids].reset_index()

        # 5) Overwrite identifiers / fields vectorized
        base_rows["artist_mbid"] = missing_with_template          # new IDs
        base_rows["mbid_mbid"] = missing_with_template            # your custom field
        base_rows["artist_name"] = "unknown"                      # same value for all

        # 6) Append and de-duplicate on artist_mbid
        new_features_df = base_rows
        nodes_df = (
            pd.concat([features_df, new_features_df], ignore_index=True)
              .drop_duplicates(subset=["artist_mbid"], keep="first")
              .reset_index(drop=True)
        )

        print(
            f"Added {len(new_features_df)} synthetic feature rows. "
            f"features_df now has {len(nodes_df)} rows."
        )
    else:
        nodes_df = features_df.copy()
        print("No synthetic rows created; no missing mbids had valid templates in features_df.")
else:
    nodes_df = features_df.copy()
    print("No synthetic rows created; no missing mbids had known neighbors.")

Can assign templates for 156471 of 156471 missing mbids.
Added 156471 synthetic feature rows. features_df now has 243233 rows.


In [37]:
nodes_df = nodes_df.drop(columns=['mbid_mbid'])

In [56]:
nodes_df[nodes_df['artist_name'].str.contains(r'nknown', na=False)]


,artist_mbid,artist_name,window_years,debut_date,window_cutoff_date,releases_total,releases_per_year,avg_days_between_releases,release_velocity_releases_per_day,release_velocity_releases_per_year,...,primary_genre,all_genres_str,artist_country,artist_region_city,years_active,debut_year,debut_decade,recency_index,primary_role,all_roles_str
6110,6b330644-d34f-4585-9405-2f9b3e11889b,Unknown Land,5,2016-03-30,2021-03-30,1,0.200027,NaN,NaN,NaN,...,NaN,NaN,Australia,NaN,9.675565,2016,2010,9.077388e-06,unknown,NaN
19253,9da1a728-773c-40cf-9807-8d5607c1d5a0,Unknown Error,5,2004-12-13,2009-12-13,23,4.600630,68.727273,0.014656,5.353286,...,NaN,NaN,United Kingdom,NaN,20.969199,2004,2000,3.802000e-05,producer,"producer, remixer"
20451,a688dc7a-c743-4ec2-9355-c3dcdb30bf0e,Unknown T,5,2022-01-14,2027-01-14,10,2.575811,28.000000,0.030967,11.310539,...,NaN,NaN,London,Homerton,3.882272,2022,2020,2.169906e-02,composer,"composer, performer, vocal, writer"
20725,921a4d9a-8a41-413d-92c1-1821da6ff296,Unknown Source,5,2003-01-01,2008-01-01,11,2.200301,146.100000,0.007198,2.629201,...,NaN,NaN,NaN,NaN,22.918549,2003,2000,1.384663e-10,unknown,NaN
22181,55e39f23-29f5-417f-92b7-d87cce3e8cd1,unknown self,5,2024-02-27,2029-02-27,3,1.701475,102.000000,0.009370,3.422541,...,NaN,NaN,Norway,NaN,1.763176,2024,2020,2.356361e-01,unknown,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243228,d1053a68-5314-4b83-96ae-5a7278253c3d,unknown,5,2003-05-04,2008-05-04,8,1.599343,223.714286,0.003632,1.326629,...,NaN,NaN,Japan,Yokohama,22.581793,2003,2000,7.841922e-01,composer,"composer, editor, instrument, lyricist, mix, p..."
243229,9a316546-ffcc-4eb6-8962-b87fd9a9e422,unknown,5,1982-01-01,1987-01-01,5,1.000137,182.500000,0.004892,1.786937,...,NaN,NaN,Wales,NaN,43.917864,1982,1980,2.481253e-04,composer,"composer, instrument, lyricist, producer, writer"
243230,c4844bd5-2927-402a-b265-c2bde04649b6,unknown,5,1971-01-01,1976-01-01,13,2.600356,121.750000,0.007117,2.599312,...,NaN,NaN,Ukraine,Chernivets'ka Oblast',54.918549,1971,1970,8.797470e-01,performer,"performer, vocal"
243231,12da7847-8a96-4d9e-8a90-85a6aa20fdd0,unknown,5,1917-01-01,1922-01-01,1,0.200027,NaN,NaN,NaN,...,NaN,NaN,France,Montargis,108.917180,1917,1910,8.597471e-01,performer,"performer, vocal"


In [43]:
edges_df.shape

(1426829, 3)

In [39]:
nodes_df.to_csv('data/graphs/only_connected/nodes.csv')

In [42]:
edges_df.to_csv('data/graphs/only_connected/edges.csv')

In [41]:
unique_mbids = set(edges_df["u"]).union(edges_df["v"])
known_mbids = set(nodes_df["artist_mbid"])
missing_mbids = unique_mbids - known_mbids

len(unique_mbids), len(known_mbids), len(missing_mbids)

(243233, 243233, 0)

In [21]:
features_df.columns

Index(['artist_mbid', 'artist_name', 'window_years', 'debut_date',
       'window_cutoff_date', 'releases_total', 'releases_per_year',
       'avg_days_between_releases', 'release_velocity_releases_per_day',
       'release_velocity_releases_per_year', 'gap_median_days', 'gap_std_days',
       'max_dry_spell_days', 'front_loading_index', 'tracks_total',
       'collab_track_rate', 'unique_collaborator_count',
       'label_diversity_count', 'label_churn', 'label_hhi', 'primary_label',
       'all_labels_str', 'duration_ms_mean', 'duration_ms_median',
       'duration_ms_min', 'duration_ms_max', 'remix_rate', 'acoustic_rate',
       'genre_count', 'genre_entropy', 'primary_genre', 'all_genres_str',
       'artist_country', 'artist_region_city', 'years_active', 'debut_year',
       'debut_decade', 'recency_index', 'primary_role', 'all_roles_str'],
      dtype='object')

In [15]:
enriched_nodes = features_df[features_df['artist_mbid'].isin(unique_mbids)]
enriched_nodes.shape

(86762, 40)

In [52]:
enriched_nodes.to_csv('data/graphs/only_connected/enriched_nodes.csv')

In [51]:
edges_df.weight.value_counts()

weight
1      18243
2       5240
3       3657
4       2410
5       1239
       ...  
186        1
222        1
127        1
79         1
74         1
Name: count, Length: 187, dtype: int64

In [6]:
edges_df.columns

Index(['u', 'v', 'weight_raw', 'weight_size_adj', 'first_collab_date',
       'last_collab_date', 'collab_span_years', 'recency_weight',
       'same_primary_genre', 'same_country', 'same_region_city',
       'same_primary_label', 'roles_overlap', 'genres_overlap',
       'labels_overlap', 'mean_team_size_cowrite_joint'],
      dtype='object')

In [7]:
with open('data/artist_collab_data/top_1000_artist_data_full.json') as f:
    data = json.load(f)

In [8]:
def filter_to_n_songs(artists, n=5):
    for artist in artists:
        artist['works'] = artist['works'][:n]
    return artists
filtered_artists = filter_to_n_songs(data)

In [15]:
nodes_df.columns

Index(['artist_mbid', 'artist_name', 'window_years', 'debut_date',
       'window_cutoff_date', 'releases_total', 'releases_per_year',
       'avg_days_between_releases', 'release_velocity_releases_per_day',
       'release_velocity_releases_per_year', 'gap_median_days', 'gap_std_days',
       'max_dry_spell_days', 'front_loading_index', 'tracks_total',
       'collab_track_rate', 'unique_collaborator_count',
       'label_diversity_count', 'label_churn', 'label_hhi', 'primary_label',
       'all_labels_str', 'duration_ms_mean', 'duration_ms_median',
       'duration_ms_min', 'duration_ms_max', 'remix_rate', 'acoustic_rate',
       'genre_count', 'genre_entropy', 'primary_genre', 'all_genres_str',
       'artist_country', 'artist_region_city', 'years_active', 'debut_year',
       'debut_decade', 'recency_index', 'primary_role', 'all_roles_str',
       'missing_recordings_flag', 'missing_genres_flag', 'missing_labels_flag',
       'missing_releases_flag', 'recency_index_missing_flag',
 

In [9]:
data[0]

{'mbid': 'c8b03190-306c-4120-bb0b-6f2ebfc06ea9',
 'artist_name': 'The Weeknd',
 'roles': 'arranger, composer, instrument, lyricist, performer, producer, programming, remixer, vocal, writer',
 'aliases': [],
 'country': 'Canada',
 'region_city': 'Scarborough',
 'works': [{'id': 420723,
   'name': 'Take On Me',
   'release_date': '1984-01-01',
   'genres': ['pop'],
   'collaborators': [{'mbid': '1ac73985-2cfa-433a-a6f8-379480e87179',
     'name': 'Magne Furuholmen',
     'roles': ['writer']},
    {'mbid': '2093207d-cb69-4789-866b-5ac772c9ebd3',
     'name': 'Morten Harket',
     'roles': ['writer']},
    {'mbid': '0ef9b4c4-4a41-4ad8-9c64-98c0419c0150',
     'name': 'Paul Waaktaar-Savoy',
     'roles': ['writer']}]},
  {'id': 441573,
   'name': 'Dirty Diana',
   'release_date': '1987-01-01',
   'genres': [],
   'collaborators': [{'mbid': 'f27ec8db-af05-4f36-916e-3d57f91ecf5e',
     'name': 'Michael Jackson',
     'roles': ['composer', 'lyricist']}]},
  {'id': 12631624,
   'name': 'High fo

In [10]:
import network_construction.graph_builder as gb
from importlib import reload
reload(gb)

<module 'network_construction.graph_builder' from '/home/woodbkb2/git/nepo-music/network_construction/graph_builder.py'>

In [11]:
avg_num_songs = gb.avg_songs_in_first_m_months(data, 32)
avg_num_songs

AttributeError: module 'network_construction.graph_builder' has no attribute 'avg_songs_in_first_m_months'

In [12]:
artist_song, edges, nodes = gb.build_graph_with_bipartite(data, first_m_months=32)

AttributeError: module 'network_construction.graph_builder' has no attribute 'build_graph_with_bipartite'

In [13]:
nodes.head()

NameError: name 'nodes' is not defined

In [14]:
edges.head()

NameError: name 'edges' is not defined

In [12]:
artist_song.head()

,artist_mbid,artist_name,song_id,song_name,credit_role,release_date,genre_tags,team_size,team_size_cowrite
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,420723,Take On Me,arranger,1984-01-01,pop,4,4
1,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,420723,Take On Me,composer,1984-01-01,pop,4,4
2,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,420723,Take On Me,lyricist,1984-01-01,pop,4,4
3,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,420723,Take On Me,producer,1984-01-01,pop,4,4
4,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,420723,Take On Me,vocal,1984-01-01,pop,4,4


In [13]:
nodes.columns

Index(['mbid', 'name', 'all_roles', 'all_genres',
       'first_release_date_in_window', 'last_release_date_in_window',
       'num_songs_in_window', 'num_collaborators_in_window',
       'time_in_network_years'],
      dtype='object')

In [14]:
edges.columns

Index(['u', 'v', 'weight_raw', 'weight_size_adj', 'first_collab_date',
       'last_collab_date', 'recency_weight', 'roles_overlap', 'genre_overlap'],
      dtype='object')

In [15]:
artist_song.columns

Index(['artist_mbid', 'artist_name', 'song_id', 'song_name', 'credit_role',
       'release_date', 'genre_tags', 'team_size', 'team_size_cowrite'],
      dtype='object')

In [16]:
nodes.shape

(2806, 9)

In [17]:
with open('data/from_1000_expanded_mbids.txt', 'w') as f:
    for mbid in nodes['mbid'].tolist():
        f.write(f'{mbid}\n')

In [18]:
import pandas as pd
import numpy as np

In [19]:
# cast dates
nodes['first_release_date_in_window'] = pd.to_datetime(nodes['first_release_date_in_window'])
nodes['last_release_date_in_window'] = pd.to_datetime(nodes['last_release_date_in_window'])
edges['first_collab_date'] = pd.to_datetime(edges['first_collab_date'])
edges['last_collab_date'] = pd.to_datetime(edges['last_collab_date'])
artist_song['release_date'] = pd.to_datetime(artist_song['release_date'])

# fill nulls
nodes['all_genres'] = nodes['all_genres'].fillna('')
nodes['all_roles'] = nodes['all_roles'].fillna('')

# ensure IDs are strings
nodes['mbid'] = nodes['mbid'].astype(str)
edges['u'] = edges['u'].astype(str)
edges['v'] = edges['v'].astype(str)

# drop self loops and duplicates
edges = edges.loc[edges['u'] != edges['v']].drop_duplicates(subset=['u','v'])

In [20]:
import re

In [23]:

# make sure string columns are strings (no lists/objects)
for col in ['mbid', 'name', 'all_roles', 'all_genres']:
    nodes[col] = nodes.get(col, "").astype(str).fillna("").replace("nan","")

# clean dates to ISO strings (R will parse as Date easily)
date_cols_nodes = ['first_release_date_in_window','last_release_date_in_window']
for c in date_cols_nodes:
    nodes[c] = pd.to_datetime(nodes[c], errors='coerce').dt.strftime('%Y-%m-%d')

date_cols_edges = ['first_collab_date','last_collab_date']
for c in date_cols_edges:
    edges[c] = pd.to_datetime(edges[c], errors='coerce').dt.strftime('%Y-%m-%d')

# ----- role_major from all_roles (single scalar string) -----
role_patterns = {
    'writer'     : r'(writer|composer|lyricist|librettist|scriptwriter)',
    'performer'  : r'(singer|performer|vocal|artist)',
    'production' : r'(producer|engineer)',
}

def map_role_major(role_text: str) -> str:
    rt = role_text.lower()
    if rt.strip() == "" or rt == "nan":
        return ""
    if re.search(role_patterns['writer'], rt): return "writer"
    if re.search(role_patterns['performer'], rt): return "performer"
    if re.search(role_patterns['production'], rt): return "production"
    return "other"

nodes['role_major'] = nodes['all_roles'].map(map_role_major).astype(str)

# ----- primary_genre from all_genres (first non-empty token) -----
def first_token(gen_text: str) -> str:
    if not isinstance(gen_text, str): return ""
    # split on commas or semicolons or pipes
    toks = [t.strip() for t in re.split(r'[,\|;]', gen_text) if t.strip()]
    return toks[0].lower() if toks else ""

nodes['primary_genre'] = nodes['all_genres'].map(first_token).astype(str)

# ----- optional: standardized continuous covariates for ERGM -----
for col in ['num_songs_in_window','time_in_network_years','num_collaborators_in_window']:
    if col in nodes:
        vals = pd.to_numeric(nodes[col], errors='coerce')
        nodes[f'{col}_std'] = ((vals - vals.mean()) / (vals.std(ddof=0) if vals.std(ddof=0) else 1)).fillna(0)

# ----- edges: types, undirected de-dup, endpoints exist -----
edges['u'] = edges['u'].astype(str)
edges['v'] = edges['v'].astype(str)

# drop self-loops
edges = edges.loc[edges['u'] != edges['v']].copy()

# keep only dyads where both endpoints are in nodes
node_set = set(nodes['mbid'].astype(str))
edges = edges[edges['u'].isin(node_set) & edges['v'].isin(node_set)]

# canonicalize undirected pairs (sort u,v)
uv = edges[['u','v']].astype(str).apply(lambda r: tuple(sorted(r.values)), axis=1)
edges = edges.loc[~uv.duplicated()].copy()

# numeric edge attrs
for col in ['weight_raw','weight_size_adj','recency_weight','roles_overlap','genre_overlap']:
    if col in edges:
        edges[col] = pd.to_numeric(edges[col], errors='coerce')

# optional weak-tie flag baked in (so R can just use edgecov("low_overlap"))
edges['low_overlap'] = ((edges['roles_overlap'] < 0.5) & (edges['genre_overlap'] < 0.5)).astype(int)

# --- final: write clean CSVs that R can ingest with no further munging ---
nodes_cols = [
    'mbid','name','all_roles','role_major','all_genres','primary_genre',
    'first_release_date_in_window','last_release_date_in_window',
    'num_songs_in_window','num_songs_in_window_std',
    'num_collaborators_in_window','num_collaborators_in_window_std',
    'time_in_network_years','time_in_network_years_std'
]
nodes_clean = nodes.reindex(columns=[c for c in nodes_cols if c in nodes])

edges_cols = [
    'u','v','weight_raw','weight_size_adj',
    'first_collab_date','last_collab_date',
    'recency_weight','roles_overlap','genre_overlap','low_overlap'
]
edges_clean = edges.reindex(columns=[c for c in edges_cols if c in edges])

In [24]:
nodes_clean.to_csv("graphs/initial_graph/nodes.csv", index=False)
edges_clean.to_csv("graphs/initial_graph/edges.csv", index=False)

In [26]:
nodes_clean.head()

,mbid,name,all_roles,role_major,all_genres,primary_genre,first_release_date_in_window,last_release_date_in_window,num_songs_in_window,num_songs_in_window_std,num_collaborators_in_window,num_collaborators_in_window_std,time_in_network_years,time_in_network_years_std
0,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,Staind,composer;performer;producer;writer,writer,hardcore;heavy metal;rock;thrash metal,hardcore,1979-01-01,1979-01-01,1,-0.437682,2,-0.540771,0.00,-0.271475
1,703c557f-82bb-4646-ae2f-b5a3ef3f6148,Jennifer Skillman,writer,writer,,,2012-01-01,2012-01-01,1,-0.437682,12,1.178860,0.00,-0.271475
2,25848dee-8562-4a78-b375-3a80b61da629,Wally Badarou,writer,writer,oldest work #10,oldest work #10,1990-01-01,1990-01-01,1,-0.437682,5,-0.024881,0.00,-0.271475
3,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,Carl Rosen,writer,writer,,,2017-09-15,2017-09-15,1,-0.437682,6,0.147082,0.00,-0.271475
4,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,Nolan Sipe,writer,writer,pop,pop,2021-10-15,2023-11-10,2,0.078125,5,-0.024881,2.07,-0.031691


In [27]:
edges_clean.head()

,u,v,weight_raw,weight_size_adj,first_collab_date,last_collab_date,recency_weight,roles_overlap,genre_overlap,low_overlap
0,0ef9b4c4-4a41-4ad8-9c64-98c0419c0150,1ac73985-2cfa-433a-a6f8-379480e87179,1,0.166667,1984-01-01,1984-01-01,1.0,1.0,1.000,0
1,0ef9b4c4-4a41-4ad8-9c64-98c0419c0150,2093207d-cb69-4789-866b-5ac772c9ebd3,1,0.166667,1984-01-01,1984-01-01,1.0,1.0,1.000,0
2,0ef9b4c4-4a41-4ad8-9c64-98c0419c0150,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,1,0.166667,1984-01-01,1984-01-01,1.0,0.1,0.125,1
3,1ac73985-2cfa-433a-a6f8-379480e87179,2093207d-cb69-4789-866b-5ac772c9ebd3,1,0.166667,1984-01-01,1984-01-01,1.0,1.0,1.000,0
4,1ac73985-2cfa-433a-a6f8-379480e87179,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,1,0.166667,1984-01-01,1984-01-01,1.0,0.1,0.125,1


In [16]:
df = pd.read_csv('./data/graphs/better_graph/node_features.csv')

In [17]:
df.columns

Index(['node', 'clust_local', 'triangles', 'open_wedges', 'gwesp_score',
       'degree', 'log_degree', 'eigencent', 'kcore', 'betweenness',
       'closeness', 'dist_to_hub_min', 'same_genre_share', 'same_label_share',
       'same_role_share', 'tri_within_genre', 'tri_within_label',
       'tri_within_role', 'tri_cross_genre', 'tri_cross_label',
       'tri_cross_role', 'component_size', 'open_dyads', 'weak_tie_frac',
       'community_participation'],
      dtype='object')